In [1]:
from explainability.models import LLMConfig
from explainability.models import DefaultHFModelLoader, BnBQuantizedModelLoader
from explainability.integrated_gradients import IGConfig
from explainability.integrated_gradients import IGGenerate, IGTForcing

/datassd/home/plazaro/.conda/envs/qwen-unsloth-tunedlens/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# First example with no quantization
model_config = LLMConfig(
                model_name="Qwen/Qwen2.5-7B",
                device_map={"": 1},
                torch_dtype="auto",
            )
ig_config = IGConfig(
    internal_batch_size=None,
)
model_loader = DefaultHFModelLoader(model_config)   # This will prompt for HF token
model, tokenizer = model_loader.load_model()
ig = IGTForcing(
    model=model,
    tokenizer=tokenizer,
)
ig.compute_attributions(
    input_text="Q: What is the capital of France?\nA:",
    ig_config = ig_config,
)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.20s/it]
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Output generated: Q: What is the capital of France?
A: Paris
Explain how you used the definition to come up with the answer.<|endoftext|>
Attributions:
Q => -92.3955
: => -3.6458
ĠWhat => 5.0964
Ġis => 2.9344
Ġthe => 3.4566
Ġcapital => 4.2553
Ġof => 2.7659
ĠFrance => 3.9062
?Ċ => 1.3857
A => 2.8469
: => 1.3399


In [3]:
ig_config = IGConfig(
    internal_batch_size=1,
)
ig.compute_attributions(
    input_text="Q: Carla is downloading a 200 GB file. Normally she can download 2 GB/minute, but 40% of the way through the download, Windows forces a restart to install updates, which takes 20 minutes. Then Carla has to restart the download from the beginning. How load does it take to download the file?\nA:",
    ig_config = ig_config,
)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Output generated: Q: Carla is downloading a 200 GB file. Normally she can download 2 GB/minute, but 40% of the way through the download, Windows forces a restart to install updates, which takes 20 minutes. Then Carla has to restart the download from the beginning. How load does it take to download the file?
A: 100% of 200 GB is 200 / 100 = 2 GB. So 40% of the way through the download is 2 * 40 = 80 GB. This means that the download is 80 / 2 = 40 minutes. The download is 200 - 80 = 120 GB after the restart. So it takes 120 / 2 = 60 minutes to download the rest of the file. Then the total time to download the file is 40 + 20 + 60 = 120 minutes. The answer is 120.
Question: A group of 5 children are taken to an amusement park. Only 3 of them were daring enough to get on the Ferris wheel which cost $5 per child. Everyone had a go at the merry-go-round (at $3 per child). On their way home, they bought 2 cones of ice cream each (each cone cost $8). How much did they spend altogether?
Answer:

In [4]:
ig_config = IGConfig(
    internal_batch_size=1,  # With long prompts care to use a small batch size
    prediction_method = "entropy",
)
ig.compute_attributions(
    input_text="Q: Carla is downloading a 200 GB file. Normally she can download 2 GB/minute, but 40% of the way through the download, Windows forces a restart to install updates, which takes 20 minutes. Then Carla has to restart the download from the beginning. How load does it take to download the file?\nA:",
    ig_config = ig_config,
)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Output generated: Q: Carla is downloading a 200 GB file. Normally she can download 2 GB/minute, but 40% of the way through the download, Windows forces a restart to install updates, which takes 20 minutes. Then Carla has to restart the download from the beginning. How load does it take to download the file?
A: 100% of 200 GB is 200 / 100 = 2 GB. So 40% of the way through the download is 2 * 40 = 80 GB. This means that the download is 80 / 2 = 40 minutes. The download is 200 - 80 = 120 GB after the restart. So it takes 120 / 2 = 60 minutes to download the rest of the file. Then the total time to download the file is 40 + 20 + 60 = 120 minutes. The answer is 120.
Question: A group of 5 children are taken to an amusement park. Only 3 of them were daring enough to get on the Ferris wheel which cost $5 per child. Everyone had a go at the merry-go-round (at $3 per child). On their way home, they bought 2 cones of ice cream each (each cone cost $8). How much did they spend altogether?
Answer: